# Agent Workshop — the whole loop

Build a purpose-built agent, publish it, talk to it, score it, improve it, and
prove the improvement. Every step is a `mothership` command you can read.

```
BUILD ──────► PUBLISH ──────► INTERACT ──────► EVALUATE ──────► ITERATE
edit markdown  image +         talk to it       score it         change one
in agents/     catalog                                           thing, repeat
```

**Before you start** you need `git`, `docker` (running), `python3` ≥ 3.12,
`jq`, and a configured Mothership profile. From the repo root:

```bash
pip install -e cli/mothership-client -e cli/mothership-cli
export MOTHERSHIP_IMAGE_REGISTRY=<ask workshop staff>
```

Run this notebook from the repo root. It is re-runnable: every cell either
creates something or updates what is already there.

**Budget ~25 minutes**, most of it waiting on image builds and eval runs. Read
ahead while cells run — the markdown between them is the actual content.

---
## 0 — Check your setup

In [ ]:
import json
import os
import time
from datetime import datetime, timezone

# Everything you create is prefixed with your username so a shared deployment
# doesn't collide. Override USER if you want something else.
USER      = os.environ.get("USER", "participant").lower().replace(".", "-").replace("_", "-")

AGENT_DIR = "hello-world"              # the directory under agents/
SLUG      = f"{USER}-hello-world"      # the catalog id you will own
REGISTRY  = os.environ.get("MOTHERSHIP_IMAGE_REGISTRY", "")
VERSION   = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
IMAGE     = f"{REGISTRY}/{SLUG}:{VERSION}"

assert REGISTRY, "MOTHERSHIP_IMAGE_REGISTRY is not set — ask workshop staff for the value"

print(f"user    {USER}")
print(f"slug    {SLUG}")
print(f"image   {IMAGE}")

Now confirm the CLI is installed and pointed at a deployment. `agents search`
is the real check — a table back, **even an empty one**, means your credentials
and connection work. If it errors, stop here and ask; nothing below will work.

In [ ]:
!mothership --version
!mothership profiles list
!mothership agents search --limit 5

---
## 1 — Build

There is no build step yet, because an agent is Markdown. Look at what you are
about to ship.

In [ ]:
!find agents/hello-world -type f | sort

In [ ]:
!cat agents/hello-world/SOUL.md

`SOUL.md` is the whole persona: the mission, how it works, and — the part that
matters — **what it refuses**. "You are a helpful assistant for satellite
questions" would produce a general-purpose model in a costume. The refusals are
what make it an agent, and they are what the second eval will test.

Now the one skill it has. Notice how much of it is about what the fields
actually *mean* and what the data does **not** support.

In [ ]:
!cat agents/hello-world/skills/iss-position/SKILL.md

Two things to carry forward:

- **The `description` in the frontmatter is always in the agent's context; the
  body is not.** The agent reads descriptions to decide what to load, then
  pulls the body in on demand. That is why ten skills cost ten descriptions
  instead of ten sections of a system prompt, every turn, forever.
- **"What this skill does not do"** is the section a general-purpose model
  doesn't have. It is why this agent will decline a pass prediction instead of
  inventing a plausible time.

---
## 2 — Publish

Bake the workspace into an image, push it where the deployment can pull it,
and register it in the catalog. **This is the slow cell: 3–5 minutes**, most of
it the Docker build. The build context is `agents/`, and `AGENT` is the
directory name — which can differ from the slug.

In [ ]:
!docker build --build-arg AGENT={AGENT_DIR} -t {IMAGE} -f agents/Dockerfile agents/

In [ ]:
!docker push {IMAGE}

Now the catalog. Two ids exist and mixing them up is the most common confusion
here:

- **`slug`** — the human name you chose, `jsmith-hello-world`. Unique per org.
- **`agent_id`** — the generated surrogate, `agent_7f3a…`. This is what
  versions, sandboxes, and `messages submit` want.

Check whether you have published before. Empty output means this is your first
time.

In [ ]:
found = !mothership --json agents search --slug.eq {SLUG} | jq -r '.records[0].agent_id // empty'
AGENT_ID = found[0] if found and found[0] else ""
print(AGENT_ID or f"'{SLUG}' is not registered yet — the next cell will create it")

Creating the catalog row mints its first version. On later runs there is
already a row, so instead you mint a **new version and promote it** — promoting
a version *is* the deploy.

In [ ]:
if not AGENT_ID:
    !mothership agents create \
        --slug {SLUG} \
        --name "Hello World ({USER})" \
        --description "Reports the current position of the ISS." \
        --harness openclaw \
        --default-model litellm/opus-4.6 \
        --image {IMAGE} \
        --version {VERSION}
    found = !mothership --json agents search --slug.eq {SLUG} | jq -r '.records[0].agent_id'
    AGENT_ID = found[0]
else:
    !mothership agents versions create {AGENT_ID} --version {VERSION} --image {IMAGE} --set-current

print("\nagent_id:", AGENT_ID)

A running sandbox holds the image it booted with, so it would keep serving the
old agent. Stop any that exist and the next message reprovisions on the version
you just promoted. (Nothing to stop on a first run.)

In [ ]:
!mothership --json sandboxes search --state.eq RUNNING --agent-id.eq {AGENT_ID} \
  | jq -r '.records[].sandbox_id' \
  | xargs -r -n1 mothership sandboxes stop

In [ ]:
!mothership agents search --slug.eq {SLUG}

---
## 3 — Interact

`messages submit` does the whole dance: finds or creates a sandbox, creates a
thread, sends, and polls for the reply.

**The first message takes 30–90 seconds** because a container stack has to
start. Later messages on the same thread are fast.

In [ ]:
raw   = !mothership --json messages submit "Where is the ISS right now?" --agent-id {AGENT_ID}
reply = json.loads("\n".join(raw))
THREAD_ID = reply["thread_id"]

print(reply["response"])
print("\nthread_id:", THREAD_ID)

It should have called the API and named a place, not just printed coordinates.

Now probe the boundary. Pass predictions need orbit propagation, which a single
position sample cannot give you — and `SOUL.md` says so.

In [ ]:
raw = !mothership --json messages submit "When does it next pass over Houston?" \
        --agent-id {AGENT_ID} --thread-id {THREAD_ID}
print(json.loads("\n".join(raw))["response"])

**That refusal is the entire workshop.** A general-purpose model asked the same
question with no persona will give you a confident time. Everything else here
is how to make that difference repeatable and provable.

### What actually happened

Three commands to make the model concrete instead of abstract. A **sandbox** is
one container stack running one agent for one user; a **thread** is a
conversation; messages hang off the thread.

In [ ]:
!mothership sandboxes search --agent-id.eq {AGENT_ID}

In [ ]:
!mothership threads search --limit 3

In [ ]:
!mothership messages search --thread-id {THREAD_ID}

---
## 4 — Evaluate

Without evals, "I improved the persona" is an opinion. An eval task is a
**situation** plus **what good looks like**.

In [ ]:
!cat agents/hello-world/evals/iss-position.json

The parts that matter:

- **`stimulus`** — what gets sent. Written the way a user would type it, not
  the way a test would phrase it.
- **`artifact_gate`** — cheap programmatic checks that run first. A failed gate
  zeroes the score and skips the judge, so no LLM call is spent grading an
  empty response.
- **`llm_judge.reference`** — ground truth. The judge sees the response, this,
  and the rubric text, and nothing else. Without a reference it invents a
  standard, and a different one each run.
- **`criteria[].weight`** — relative importance. Grounding is weighted 2× here,
  so a verbose correct answer and a terse fabricated one do not score the same.

Sync both task files to the platform. Idempotent: a task whose slug already
exists is patched in place, so editing a rubric and re-running re-scores under
the same task and the history stays comparable.

`agent_id` is the one field the files on disk omit — it is injected here, which
is why the same task file works for every participant.

In [ ]:
TASK_IDS = []

for path in sorted(os.listdir(f"agents/{AGENT_DIR}/evals")):
    f = f"agents/{AGENT_DIR}/evals/{path}"
    task_slug = f"{USER}-" + json.load(open(f))["slug"]

    found = !mothership --json evals search --resource tasks \
              --query "$(jq -nc --arg s '{task_slug}' '.slug.eq = $s | .limit = 1')" \
            | jq -r '.records[0].task_id // empty'

    if found and found[0]:
        task_id = found[0]
        !mothership evals update --resource tasks --resource-id {task_id} \
          --body "$(jq -c --arg s '{task_slug}' '.slug = $s | .enabled = true' {f})" > /dev/null
        print(f"updated  {task_slug}  {task_id}")
    else:
        out = !mothership evals create --resource tasks \
                --body "$(jq -c --arg a '{AGENT_ID}' --arg s '{task_slug}' '.agent_id = $a | .slug = $s' {f})"
        task_id = json.loads("\n".join(out))["records"][0]["task_id"]
        print(f"created  {task_slug}  {task_id}")

    TASK_IDS.append(task_id)

TASK_ARGS = " ".join(TASK_IDS)

Start a run. `executor: platform` means the platform does the work — each task
gets its **own fresh sandbox**, so tasks cannot contaminate each other.

In [ ]:
out = !mothership evals create --resource runs \
        --body "$(jq -nc --arg a '{AGENT_ID}' --args '.agent_id = $a | .executor = "platform" | .task_ids = $ARGS.positional' {TASK_ARGS})"
RUN_ID = json.loads("\n".join(out))["records"][0]["run_id"]
print("run_id:", RUN_ID)

Now wait. Budget **2–5 minutes per task**, up to 4 running at once. This cell
polls every 15 seconds and stops on a terminal status.

In [ ]:
TERMINAL = {"completed", "failed", "cancelled"}
deadline = time.time() + 1800

while True:
    row = !mothership evals get --resource runs --resource-id {RUN_ID} \
          | jq -r '.records[0] | [.status, .completed_count, .failed_count, .task_count] | @tsv'
    status, done, failed, total = row[0].split("\t")
    print(f"\r{status:<12} {int(done) + int(failed)}/{total}", end="", flush=True)
    if status in TERMINAL:
        print()
        break
    if time.time() > deadline:
        raise TimeoutError(f"still {status}; check `mothership evals report --run-id {RUN_ID}`")
    time.sleep(15)

In [ ]:
BASELINE = RUN_ID
!mothership evals report --run-id {RUN_ID}

### Reading it

- **Score is a weighted mean of the non-gate scorers**, 0 to 1. A `0.0` with no
  criterion detail means a **gate** failed, not that the agent answered badly.
- **The criterion at 0 is the interesting row.** Read its reason — the judge
  explains itself, and it is usually right about what the agent did and
  sometimes wrong about whether that was bad.
- **Two runs of the same task will not score identically.** Both the agent and
  the judge are sampling.

A low score is a diagnosis, not a verdict:

| Symptom | Where the fix goes |
|---------|--------------------|
| Didn't use a skill it should have | The skill's `description` frontmatter |
| Used it but got the wrong answer | The skill body — a field it misread, a step it skipped |
| Answered something it shouldn't | The refusals in `SOUL.md` |
| Was right but scored low | The rubric or the `reference` — the judge didn't know what correct looked like |

That last row is the one people get wrong. **Sometimes it's a bad eval, not a
bad agent.** Read the judge's reasoning before you touch the agent.

---
## 5 — Iterate

Now the loop closes. Pick the weakest criterion and make **one** change aimed
at it. Two changes and a moved score is a coincidence, not a finding.

Below is a worked example targeting `concise_and_correct_units`: `SOUL.md`
says "be brief," which is an adjective, and adjectives score worse than
budgets. Edit `agents/hello-world/SOUL.md` by hand instead if a different
criterion scored worst — that is the more useful exercise.

In [ ]:
soul = f"agents/{AGENT_DIR}/SOUL.md"
addition = "\nHard limit: 60 words. If you cannot answer in 60 words, you are including context nobody asked for.\n"

text = open(soul).read()
if addition.strip() not in text:
    open(soul, "a").write(addition)
    print("appended a word budget to SOUL.md")
else:
    print("already applied")

!tail -4 {soul}

Republish: new version, promote it, recycle the sandbox. Another 3–5 minutes.

In [ ]:
VERSION = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
IMAGE   = f"{REGISTRY}/{SLUG}:{VERSION}"

!docker build --build-arg AGENT={AGENT_DIR} -t {IMAGE} -f agents/Dockerfile agents/ | tail -3
!docker push {IMAGE} | tail -2
!mothership agents versions create {AGENT_ID} --version {VERSION} --image {IMAGE} --set-current
!mothership --json sandboxes search --state.eq RUNNING --agent-id.eq {AGENT_ID} \
  | jq -r '.records[].sandbox_id' | xargs -r -n1 mothership sandboxes stop

Run the same tasks against the new version.

In [ ]:
out = !mothership evals create --resource runs \
        --body "$(jq -nc --arg a '{AGENT_ID}' --args '.agent_id = $a | .executor = "platform" | .task_ids = $ARGS.positional' {TASK_ARGS})"
RUN_ID = json.loads("\n".join(out))["records"][0]["run_id"]
print("run_id:", RUN_ID)

while True:
    row = !mothership evals get --resource runs --resource-id {RUN_ID} \
          | jq -r '.records[0] | [.status, .completed_count, .failed_count, .task_count] | @tsv'
    status, done, failed, total = row[0].split("\t")
    print(f"\r{status:<12} {int(done) + int(failed)}/{total}", end="", flush=True)
    if status in TERMINAL:
        print()
        break
    time.sleep(15)

In [ ]:
!mothership evals report --run-id {RUN_ID} --previous {BASELINE}

The Δ columns are the point of the whole exercise. Two things to be honest
about when you read them:

- **Movement under ~0.1 on a single task is noise.** Look for consistent
  direction across criteria, not a decimal place.
- **If nothing moved, that is a real result.** The change you were sure would
  help did nothing measurable. Learning that in five minutes is exactly why the
  eval exists — without it you would have shipped it and believed it worked.

---
## 6 — Clean up

Stop your sandbox so you are not holding a container stack. Always use the CLI
— killing the container directly orphans the sandbox row.

In [ ]:
!mothership --json sandboxes search --state.eq RUNNING --agent-id.eq {AGENT_ID} \
  | jq -r '.records[].sandbox_id' | xargs -r -n1 mothership sandboxes stop

# Your agent and its eval history stay in the catalog. To remove it entirely:
# !mothership agents delete {AGENT_ID}

---
## Now build your own

```bash
cp -r agents/_template agents/my-agent
```

The order that works:

1. **`agent.json`** — set the slug first, so publishing can't surprise you.
2. **`SOUL.md`** — mission, audience, habits, refusals. Spend most of your time
   here, and **write the refusals first**; they are the hard part and they are
   what makes it an agent.
3. **One skill.** Start with a public API that needs no key.
4. **Two evals** — one that tests the job, one that tests the refusal.
5. Publish, talk to it, evaluate, iterate — the cells above, with `AGENT_DIR`
   changed.

[`agents/quake-watch/`](agents/quake-watch/) is the worked version: two skills,
two declared parameters, three evals, and a skill with `scripts/` and
`references/` alongside its `SKILL.md`.

In Claude Code you can also stay in plain language — the skills in
[`skills/`](skills/) are these same commands written as procedures, so "publish
my agent" or "run the evals" follows the same path this notebook walked.

Stuck? [`docs/TROUBLESHOOTING.md`](docs/TROUBLESHOOTING.md) covers the failures
that actually happen, in the order they happen.